In [3]:
import geemap
import utils as F
import ee
import os
import concurrent.futures
import glob
import rasterio
from rasterio.merge import merge
import geopandas as gpd
from shapely.geometry import mapping
from tqdm import tqdm
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np

In [17]:
Q = geemap.Map()

In [28]:
roi = Map.user_roi
year = 2022
start_date, end_date = '2019-01-01', '2020-01-01'
MGRS_TILE = '30UYD'
cloud_limit = 40
S2 = F.get_s2_sr_cld_col(roi, start_date, end_date, 40)

def CLIP(img):
    return img.clip(roi)
def cal_cloud_per(img):
    
    cloud_img = img.select('cloud')
    Cloud_count = cloud_img.reduceRegion(
      
      reducer = ee.Reducer.sum(),
      geometry = roi,
      scale = 30,
      maxPixels = 1e13
    ).get('cloud')

    count = cloud_img.reduceRegion(
      
      reducer = ee.Reducer.count(),
      geometry = roi,
      scale = 30,
      maxPixels =  1e13
    ).get('cloud')
    
    cloud_per = ee.Number(Cloud_count).divide(count).multiply(100)
    return img.set('cloud_per', cloud_per)

def cal_TF_per(img):
    
    cloud_img = img.select('TF')
    Cloud_count = cloud_img.reduceRegion(
      
      reducer = ee.Reducer.sum(),
      geometry = roi,
      scale = 30,
      maxPixels = 1e13
    ).get('TF')

    count = cloud_img.reduceRegion(
      
      reducer = ee.Reducer.count(),
      geometry = roi,
      scale = 30,
      maxPixels =  1e13
    ).get('TF')
    
    cloud_per = ee.Number(Cloud_count).divide(count).multiply(100)
    return img.set('TF', cloud_per)


S2 = S2.filter(ee.Filter.eq('MGRS_TILE', MGRS_TILE))\
            .map(F.VI_cal_S2).map(F.add_cloud_bands)\
            .map(CLIP).map(F.TF_get).map(cal_cloud_per).map(cal_cloud_per)\
            .filter(ee.Filter.gt('cloud_per', cloud_limit))


In [ ]:
TFI_vis = {'palette': ['0000ff', '00ffff', 'ffff00', 'ff0000', 'ffffff'],'min': 0,'max': 1.5}
viz = {'bands':['B8', 'B4', 'B3'], 'min':0, 'max':4000}
Q.addLayer(S2.first().select('TFI'), TFI_vis, 'tfi')
Q.addLayer(S2.first(), viz, 'img')

In [19]:
Q

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…